# UPRM Hackathon 2026 - Object Detection using Convolutional Neural Networks

**Author:** Hamzah Abdulrazzaq, Chelsea Cantone, Erika Hall, Edgar Perez

**Date:** September 25-27, 2026

**Objective:** UPRM Hackathon 2026 will be focused on exploring the capabilities of Convolutional Neural Networks (CNNs) in tackling complex object detection tasks. Specifically, participants will delve into the realm of single-class object detection using the 'Remondis Contamination Dataset' (RCD), which comprises truck-mounted waste camera imagery captured in real-world waste management operations. The dataset includes KITTI-format annotated images featuring plastic_bag instances of varying sizes, occlusion levels, lighting conditions, and background clutter, with bounding-box labels marking each target object. The task at hand is to design and optimize CNN-based detection architectures to accurately localize and classify plastic bags within a given image. The challenge lies in developing a robust model that can effectively handle the complexities of bounding-box regression combined with classification, requiring meticulous attention to data preprocessing, hyper-parameter tuning, and potentially incorporating techniques such as transfer learning, data augmentation, anchor tuning, and ensemble methods. The winning submissions will be evaluated based on their performance, with a key metric being the model's ability to achieve high mean Average Precision at IoU 0.50 (mAP@50) on the validation split, balancing precision and recall across detected instances.

## Table of Contents
1. [Background Information](#1-background-information)
    - [1.1. Convolutional Neural Networks](#11-convolutional-neural-networks)
    - [1.2. Object Detection vs Image Classification (Multi-label)](#12-object-detection-vs-image-classification-multi-label)
    - [1.3 An Example Use Case of Object Detection for Lockheed Martin](#13-an-example-use-case-of-object-detection-for-lockheed-martin)
    - [1.4 Key Terms](#14-key-terms)
    - [1.5 Helpful Resources](#15-helpful-resources)
2. [Setup](#2-setup)
    - [2.1. Download Data](#21-download-data)
    - [2.2 Importing Necessary Libraries](#22-importing-necessary-libraries)
    - [2.3 Custom Image Dataset](#23-custom-image-dataset)
    - [2.4 Image Augmentation](#24-image-augmentation)
3. [Training Loop](#3-training-loop)
    - [3.1 CNN Model Architecture](#31-cnn-model-architecture)
    - [3.2 Model Training](#32-model-training)
4. [Analysis](#4-analysis)
    - [4.1 Visualize Ground Truth vs Predictions](#41-visualize-ground-truth-vs-predictions)
    - [4.2 Performance Analysis (Where is the model underperforming)](#43-performance-analysis-where-is-the-model-underperforming)
5. [Test](#5-test)
    - [5.1 Download Test Data](#51-download-test-data)
    - [5.2 Create Test DataLoader](#51-create-test-dataloader)
    - [5.3 Grade Model on Test Set](#52-grade-model-on-test-set)
6. [Grading/Submission Guidelines](#6-gradingsubmission-guidelines)

## 1. Background Information

### 1.1 Convolutional Neural Networks:

A convolutional neural network (CNN or ConvNet) is a class of deep neural networks, most commonly applied to analyzing visual imagery. They have proven to be very effective in image and video recognition, recommender systems, image classification, medical image analysis, natural language processing, brain-computer interfaces, and financial time series modeling. CNNs are regularized versions of multilayer perceptrons. Multilayer perceptrons usually mean fully connected networks, that is, each neuron in one layer is connected to all neurons in the next layer. The "fully-connectedness" of these networks makes them prone to overfitting data. Typical ways of regularization include adding some form of magnitude measurement of weights to the loss function. CNNs take a different approach towards regularization: they take advantage of the hierarchical pattern in data and assemble more complex patterns using smaller and simpler patterns. Therefore, on the scale of connectedness and complexity, CNNs are on the lower extreme. Convolutional networks were inspired by biological processes in that the connectivity pattern between neurons resembles the organization of the animal visual cortex. Individual cortical neurons respond to stimuli only in a restricted region of the visual field known as the receptive field. The receptive fields of different neurons partially overlap such that they cover the entire visual field.

### 1.2 Object Detection vs Image Classification (Multi-label)

![choose_computer_vision_arch.png](./img/choose_computer_vision_arch.png)

*[Image Source](https://storage.ghost.io/c/2c/8d/2c8d8c0d-1c15-4b6d-825e-02b78d61d40a/content/images/size/w1000/size/w2000/2020/07/choose_computer_vision_arch.png)*

Object detection vs image classification are two related but different challenges in computer vision. Object detection looks at digital images and videos to locate objects of interest and label them.

| Image Classification (multi-label) | Object Detection |
| :--- | :--- |
| *Questions*: What labels are present in this image? | *Questions*: Is there an object here (objectness)? Where are the objects in the image (localization)? What labels are present (classification)? |
| *Output*: Class probabilities | *Output*: Bounding boxes, class probabilities, confidence scores (objectness) |
| *Evaluation metric*: F1-score | *Evaluation metric*: mAP@50 IoU, mean average precision at 50% Intersection over Union |

For explanation of mAP, see the references section below. mAP@50 indicates the mean Average Precision, where the Intersection over Union threshold is set to 0.5. Practically this means that the overlap of the ground truth bounding box to the predicted bounding box sufficiently overlap.

![Intersection over Union](./img/IoU-graphic.png)

[*Source*](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*FrmKLxCtkokDC3Yr1wc70w.png)

### 1.3 An Example Use Case of Object Detection for Lockheed Martin
> GATR (Globally-scalable Automated Target Recognition) is a Lockheed Martin software system for real-time object detection and classification in satellite imagery on a worldwide basis. 
![GATR Object Detection](./img/Screenshot%202026-08-10%20134544.png)

[*Globally-scalable Automated Target Recognition (GATR) 2019*](https://arxiv.org/pdf/2009.04836)

### 1.4 Key Terms:
- **Anchor box** - in some object detection CNN models predefined reference boxes, of fixed size and aspect ratio, are used to localize objects
- **Backbone** - a backbone is a very unspecialized CNN that is inserted into a computer vision model to extract essential visual features from raw images (edges, corners, textures, shapes, etc.) into a feature map. Example backbone are ResNet, CSPNet, or a Vision Transformer.
- **Bounding box (bbox)** - the rectangular frame surrounding an object in an image or video. Our labeled dataset contains ground truth bounding boxes that were annotated by humans. We will compare the bounding boxes generated by our model to the ground truth annotations.
- **Localization** - uses regression to predict four numbers representing bounding box coordinates (center point, width, and height).
- **Objectness** - in computer vision, measure that scores how likely a specific image window contains an actual object rather than background.
- **Non-Maximum Suppression (NMS)** - A post processing technique to remove overlapping bounding boxes of the same objecct. Some computer vision models do not deal with this challenge.


### 1.5 Helpful Resources:

Neural Networks:
- [What are Convolutional Neural Networks](https://www.bing.com/videos/search?q=convolutional+neural+network&docid=603549927117705930&mid=E7ED04059A9474B24BFEE7ED04059A9474B24BFE&view=detail&FORM=VIRE)
- [CNN Explainer - Learn CNN In Your Browser](https://poloclub.github.io/cnn-explainer/)
- [Neural Network Playground](https://playground.tensorflow.org/)
- [Interactive Demo for Image kernels](https://deeplizard.com/resource/pavq7noze2)
- [Introduction to Pooling Layers](https://www.geeksforgeeks.org/deep-learning/cnn-introduction-to-pooling-layer/)

Object Detection
- [Overview of Object Detection](https://machinelearningmastery.com/object-recognition-with-deep-learning/)
- [Anchor-Based vs Anchor-Free Object Detection](https://www.abhik.ai/concepts/computer-vision/anchor-based-vs-anchor-free)
- [YOLO Object Detection Explained](https://www.datacamp.com/blog/yolo-object-detection-explained)
- [mAP (mean Average Precision) for Object Detection](https://jonathan-hui.medium.com/map-mean-average-precision-for-object-detection-45c121a31173)

## 2. Setup

### 2.1. Download Data

Unzip your data into the data/ folder so the folder structure looks like:

```
data/
  training/
    image/      # *.jpg
    label/      # *.txt (KITTI format)
  val/
    image/      # *.jpg
    label/      # *.txt (KITTI format)
```

### 2.2 Importing Necessary Libraries

In [ ]:
!pip install -r requirements.txt

In [ ]:
import math
import os
import random
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import boxes as box_ops
import torchvision.transforms.functional as TF
from PIL import Image

import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Repository root (assumes the notebook lives in <repo>/event_materials/).
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "event_materials" else Path.cwd()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Class id 0 is reserved for background in torchvision detection models.
CLASS_NAME_TO_ID = {"plastic_bag": 1}
NUM_CLASSES = 1 + len(CLASS_NAME_TO_ID)  # background + foreground

print(f"Using device: {DEVICE}")
print(f"NUM_CLASSES (background + foreground): {NUM_CLASSES}")
print(f"Repo root: {REPO_ROOT}")

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU detected.")
    

In [ ]:
import warnings 
warnings.filterwarnings("ignore")

### 2.3 Custom Image Dataset

A PyTorch Custom Image Dataset is a class that inherits from PyTorch's Dataset class which provides a way to create a dataset that is tailored to your requirements and needs. It integrates efficiently with PyTorch's DataLoader, allowing us to load and process images in batches during model training and validation

#### What is it used for?
- Data Loading: Standardized way to load images and labels from disk.
- Data Preprocessing: Consistent application of transformations (resizing, augmentation, etc.) to images.
- Batch Creation: Mini-batch generation for training.
- Memory Efficiency: Load images only when needed rather than storing entire dataset in memory. 

Provided below is a custom image dataset that you may use for this challenge. You may add your own modifications by reading through [PyTorch's documentation](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html). 

In [ ]:
# Directory names differ between the two splits (hard-coded on disk).
_SPLIT_DIRS = {
    "train": ("image", "label"),
    "val":   ("image",   "label"),
    "test": ("image", "label")
}


def _parse_kitti_label(label_path: Path) -> Tuple[List[List[float]], List[int]]:
    """Parse a single KITTI-format .txt file into boxes and labels.

    Returns (boxes_xyxy, labels). Boxes with non-positive width/height are dropped.
    Only foreground classes in CLASS_NAME_TO_ID are kept; others are silently ignored.
    """
    boxes: List[List[float]] = []
    labels: List[int] = []
    if not label_path.exists():
        return boxes, labels
    with open(label_path, "r") as f:
        for raw in f:
            parts = raw.strip().split()
            if len(parts) < 8:
                continue
            cls = parts[0]
            if cls not in CLASS_NAME_TO_ID:
                continue
            xmin, ymin, xmax, ymax = (float(parts[4]), float(parts[5]),
                                      float(parts[6]), float(parts[7]))
            if xmax <= xmin or ymax <= ymin:
                continue
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(CLASS_NAME_TO_ID[cls])
    return boxes, labels


class KittiPlasticBagDataset(Dataset):
    """PyTorch Dataset over KITTI-format plastic-bag data (self-contained version)."""

    def __init__(
        self,
        root,
        split: str = "train",
        transforms: Optional[Callable] = None,
    ) -> None:
        if split not in _SPLIT_DIRS:
            raise ValueError(f"split must be one of {list(_SPLIT_DIRS)}, got {split!r}")
        self.split = split
        self.transforms = transforms

        root = Path(root)
        # Accept either the repo `data/` folder or a specific split root.
        if (root / "training").is_dir() and (root / "val").is_dir():
            split_root = root / ("training" if split == "train" else split)
        else:
            split_root = root
        img_dir_name, lbl_dir_name = _SPLIT_DIRS[split]
        self.image_dir = split_root / img_dir_name
        self.label_dir = split_root / lbl_dir_name
        if not self.image_dir.is_dir():
            raise FileNotFoundError(f"Image dir not found: {self.image_dir}")
        if not self.label_dir.is_dir():
            raise FileNotFoundError(f"Label dir not found: {self.label_dir}")

        stems = sorted(p.stem for p in self.image_dir.glob("*.jpg"))
        self.ids: List[str] = [s for s in stems if (self.label_dir / f"{s}.txt").exists()]
        if not self.ids:
            raise RuntimeError(f"No paired image/label files found under {split_root}")

    def __len__(self) -> int:
        return len(self.ids)

    def __getitem__(self, idx: int):
        stem = self.ids[idx]
        img_path = self.image_dir / f"{stem}.jpg"
        lbl_path = self.label_dir / f"{stem}.txt"

        img = Image.open(img_path).convert("RGB")
        boxes, labels = _parse_kitti_label(lbl_path)

        if boxes:
            boxes_t = torch.as_tensor(boxes, dtype=torch.float32)
            labels_t = torch.as_tensor(labels, dtype=torch.int64)
        else:
            boxes_t = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": boxes_t,
            "labels": labels_t,
            "image_id": torch.tensor([idx], dtype=torch.int64),
        }

        if self.transforms is not None:
            img, target = self.transforms(img, target)
        else:
            img = TF.to_tensor(img)

        return img, target


def detection_collate_fn(batch):
    """Detection targets are variable-length; keep them as a tuple of dicts."""
    images, targets = list(zip(*batch))
    return list(images), list(targets)


### 2.4 Image Augmentation

Image augmentation is an absolutely wonderful way to artificially expand your training dataset, which is especially valuable when working with limited data — a common constraint in real-world image tasks. By creating variations of existing images through transformations like rotations, flips, zooms, and crops, you essentially teach your CNN to recognize objects under different viewing conditions without collecting additional data.

More fundamentally, augmentation directly addresses overfitting by preventing the network from memorizing exact pixel patterns. When a CNN sees the same exact images repeatedly, it tends to learn specific pixel configurations rather than generalizable features. By presenting slightly different versions each time, you force the network to learn robust representations that capture the essence of objects rather than superficial details.

Augmentation also introduces invariance properties into your model. For classification tasks, whether an image is slightly rotated, zoomed, or shifted shouldn't change the label. By training with these variations, your CNN develops translation, rotation, and scale invariance—crucial properties for real-world deployment where objects rarely appear in perfectly consistent positions or orientations.

The CustomImageDataset created above is setup to handle image transforms. An example is provided below with horizontal flip and color jitter transforms applied. See documentation [here](https://docs.pytorch.org/vision/stable/transforms.html?highlight=compose)


In [ ]:
from torchvision.transforms import ColorJitter


class TrainTransforms:
    """Light on-the-fly augmentation for the training split.

    - Random horizontal flip (with box coord update).
    - Photometric color jitter (brightness/contrast/saturation/hue) applied
      to the PIL image before tensor conversion. Reduces train/val
      overconfidence gap without needing to re-map bounding boxes.
    - PIL -> float tensor in [0, 1], CHW.
    """

    def __init__(self, hflip_prob: float = 0.5) -> None:
        self.hflip_prob = float(hflip_prob)
        self.color_jitter = ColorJitter(
            brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05,
        )

    def __call__(self, img: Image.Image, target: dict) -> Tuple[torch.Tensor, dict]:
        if self.hflip_prob > 0 and random.random() < self.hflip_prob:
            w = img.width
            img = TF.hflip(img)
            if target["boxes"].numel() > 0:
                boxes = target["boxes"].clone()
                x1 = w - boxes[:, 2]
                x2 = w - boxes[:, 0]
                boxes[:, 0] = x1
                boxes[:, 2] = x2
                target["boxes"] = boxes

        # Photometric jitter operates purely on pixel intensities; boxes
        # are unchanged, so no target update is needed here.
        img = self.color_jitter(img)

        img_t = TF.to_tensor(img)
        return img_t, target

### 2.6 Datasets and DataLoader

In [ ]:
DATA_ROOT = REPO_ROOT / "data"
print("Data root:", DATA_ROOT)

train_ds = KittiPlasticBagDataset(root=DATA_ROOT, split="train", transforms=TrainTransforms(hflip_prob=0.5))
val_ds   = KittiPlasticBagDataset(root=DATA_ROOT, split="val")

print(f"train samples: {len(train_ds)}")
print(f"val   samples: {len(val_ds)}")

# Modify BATCH_SIZE as needed depending on GPU vs CPU
BATCH_SIZE = 4
# NUM_WORKERS = 0 if running on CPU
NUM_WORKERS = 4

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=detection_collate_fn,
    drop_last=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=detection_collate_fn,
    drop_last=False
)

In [ ]:
# Sanity check: folder names, file types and one example label
from collections import Counter

for split in ["training", "val"]:
    d = REPO_ROOT / "data" / split
    print(f"\n=== {split} ===")
    if not d.is_dir():
        print("  MISSING:", d)
        continue
    for sub in sorted(d.iterdir()):
        if sub.is_dir():
            exts = Counter(p.suffix.lower() for p in sub.iterdir())
            print(f"  {sub.name}/  ->  {dict(exts)}")


example = next((REPO_ROOT / "data" / "training").rglob("*.txt"), None)
print("\nExample label:", example)
if example:
    print(example.read_text()[:500])

## 3. Training Loop

### 3.1 CNN Model Architecture

There are two ways to approaching the model architecture phase of the problem. 

1. Building your own CNN architecture. An example is provided below where modifications can be made. 
2. Using pretrained models.

#### Building Your Own CNN Architecture

__Advantages:__

- Complete control over the architecture design. 
- Freedom to experiment with novel layer arrangements, activation functions, and skip connections.
- No constraints from pretrained weights or architectures that may not be optimal for your domain.
- Typically results in a smaller, more compute efficient model since you only include what you need.

__Disadvantages:__

- Requires a bit more expertise and understanding of Neural Network design
- More epochs are needed as learning must start from random initialization
- Needs much more data to achieve good performance (often prohibitively large amounts)
- Higher risk of overfitting, especially with limited data

#### Using Pretrained Models

__Advantages:__

- Leverage powerful feature extractors trained on massive datasets (like ImageNet with 1.2M images)
- Dramatically reduced training time and computational requirements (though more compute resources are needed to get started)
- Excellent performance even with limited domain-specific data
- Built-in robustness against common image variations and noise
- Access to state-of-the-art architectures (ResNet, EfficientNet, Vision Transformers) without designing them

__Disadvantages:__

- Less control over the underlying architecture
- May include unnecessary complexity for simpler tasks
- Domain shift can be problematic if your images differ significantly from pretraining data
- Fixed input requirements (resolution, channels) that may require preprocessing



#### Our Approach: Pretrained Faster R-CNN (Transfer Learning)

With only 628 training images, we chose option 2: a **Faster R-CNN (ResNet-50 + FPN v2)** from `torchvision`, pretrained on COCO, and fine-tuned on the plastic bag dataset.

- **Backbone (ResNet-50) + FPN**: extracts features at several resolutions, which helps with both small and large bags.
- **RPN (Region Proposal Network)**: proposes regions that may contain an object, using anchor boxes.
- **RoI Head**: classifies each proposal as *plastic_bag* or *background* and refines its box.
- **NMS**: removes duplicate boxes on the same object.

We replace the final classification layer (91 COCO classes) with a new one for 2 classes (background + plastic_bag). All other layers keep their pretrained weights and are fine-tuned with a small learning rate.

### 3.2 Model Training

SGD with momentum, 1 epoch of linear warmup followed by cosine decay. The cell below creates the model and optimizer and runs one forward pass as a sanity check.

In [ ]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# Faster R-CNN pretrained on COCO (weights are downloaded the first time)
model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(
    weights="DEFAULT",
    box_score_thresh=0.001,       # low threshold: keeps low-confidence boxes, helps mAP
    box_detections_per_img=100,
)
# Replace the final layer: from 91 COCO classes to 2 (background + plastic_bag)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
model.to(DEVICE)

# Training hyperparameters
EPOCHS = 12
LR = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 10.0
WARMUP_EPOCHS = 1

optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
)
# Linear warmup for the first epoch, then cosine decay
warmup = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS,
)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS - WARMUP_EPOCHS,
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS],
)

# Sanity check: one forward pass with a real training batch
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model created: {n_params:.1f}M parameters, on {next(model.parameters()).device}")

images, targets = next(iter(train_loader))
images = [img.to(DEVICE) for img in images]
targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
model.train()
loss_dict = model(images, targets)
print("Initial losses:", {k: round(v.item(), 4) for k, v in loss_dict.items()})
print(f"Peak GPU memory used: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

In [ ]:
# Calculate mAP@50 helper function
@torch.no_grad()
def evaluate_map50(model, loader, device) -> float:
    model.eval()
    all_scores: List[float] = []
    all_matches: List[int] = []  # 1 = TP, 0 = FP
    total_gt = 0
    
    # Look through each image and make a prediction
    # Compare each prediction to all ground truth boxes and
    # compute an IoU. If that IoU is >0.50 then consider it a match
    for images, targets in loader:
        images_d = [img.to(device) for img in images]
        preds = model(images_d)
        for pred, tgt in zip(preds, targets):
            gt_boxes = tgt["boxes"]
            total_gt += int(gt_boxes.shape[0])

            p_boxes = pred["boxes"].detach().cpu()
            p_scores = pred["scores"].detach().cpu()

            # Sort predictions by score descending.
            order = p_scores.argsort(descending=True)
            p_boxes = p_boxes[order]
            p_scores = p_scores[order]

            matched = torch.zeros(gt_boxes.shape[0], dtype=torch.bool)
            if gt_boxes.numel() == 0:
                for s in p_scores.tolist():
                    all_scores.append(s)
                    all_matches.append(0)
                continue

            if p_boxes.numel() == 0:
                continue
            
            # IoU calculation against all predicted boxes vs GT boxes
            # in current image
            iou = box_ops.box_iou(p_boxes, gt_boxes)  # [P, G]
            for pi in range(p_boxes.shape[0]):
                ious_p = iou[pi].clone()
                ious_p[matched] = -1.0
                best_iou, best_gt = ious_p.max(dim=0)
                if best_iou.item() >= 0.5:
                    matched[best_gt] = True
                    all_matches.append(1)
                else:
                    all_matches.append(0)
                all_scores.append(float(p_scores[pi].item()))

    if total_gt == 0 or not all_scores:
        return 0.0
    
    # Reorder matches from most to least confidence (high to low objectness)
    scores_t = torch.tensor(all_scores)
    matches_t = torch.tensor(all_matches, dtype=torch.float32)
    order = scores_t.argsort(descending=True)
    matches_t = matches_t[order]

    # Calculate average precision
    tp_cum = torch.cumsum(matches_t, dim=0) # cumulative true‑positives
    fp_cum = torch.cumsum(1 - matches_t, dim=0) # cumulative false‑positives
    recall = tp_cum / max(1, total_gt)
    precision = tp_cum / (tp_cum + fp_cum).clamp(min=1e-9)

    # 11-point interpolated AP (VOC-style).
    # The original PASCAL VOC challenge (2007‑2010) used exactly these 11 recall levels. 
    # The area under the interpolated precision‑recall curve is approximated by taking the 
    # average of the 11 sampled precisions, which yields the Average Precision (AP).
    ap = 0.0
    for t in torch.linspace(0, 1, 11):
        mask = recall >= t
        p = precision[mask].max().item() if mask.any() else 0.0
        ap += p / 11.0
    return float(ap)


In [ ]:
# Training loop for one epoch (adapted for Faster R-CNN)
def train_one_epoch(model, loader, optimizer, device, epoch: int, log_every: int = 20):
    model.train()
    running = {}
    n_batches = 0
    for it, (images, targets) in enumerate(loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Faster R-CNN returns a dict with 4 losses; we train on their sum
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        # Skip the batch if the loss becomes NaN/inf (safety check)
        if not torch.isfinite(loss):
            print(f"  invalid loss at iteration {it}, skipping batch")
            continue

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if GRAD_CLIP is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        # Accumulate losses for logging
        running["loss"] = running.get("loss", 0.0) + loss.item()
        for k, v in loss_dict.items():
            running[k] = running.get(k, 0.0) + v.item()
        n_batches += 1

        if (it + 1) % log_every == 0:
            print(f"  epoch {epoch} it {it+1}/{len(loader)}  loss={running['loss']/n_batches:.4f}")

    return {k: v / max(1, n_batches) for k, v in running.items()}

**Full training run.** Trains for `EPOCHS` epochs and evaluates on validation after each one. `last.pt` is always saved, and `best.pt` is saved whenever validation mAP@50 improves.

> ⚠️ Running this cell again overwrites the checkpoints in `models/faster_rcnn/`. Back up `best.pt` first if you want to keep it.

In [ ]:
import time

# Where checkpoints will be saved
SAVE_DIR = REPO_ROOT / "models" / "faster_rcnn"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

history = []
best_map = -1.0
total_start = time.time()

for epoch in range(1, EPOCHS + 1):
    print(f"---- Epoch {epoch}/{EPOCHS} (lr={optimizer.param_groups[0]['lr']:.5g}) ---")
    t0 = time.time()

    # Train for one epoch, then update the learning rate
    stats = train_one_epoch(model, train_loader, optimizer, DEVICE, epoch)
    scheduler.step()

    # Evaluate on the validation set
    map50 = evaluate_map50(model, val_loader, DEVICE)
    elapsed = time.time() - t0
    history.append({"epoch": epoch, "elapsed_sec": elapsed, "val_map50": map50, **stats})
    print(f"epoch {epoch}: loss={stats.get('loss', 0):.4f}  val mAP@50={map50:.4f}  ({elapsed:.0f}s)")

    # Always save the latest model; save separately when validation mAP improves
    ckpt = {
        "model_state_dict": model.state_dict(),
        "epoch": epoch,
        "val_map50": map50,
        "history": history,
    }
    torch.save(ckpt, SAVE_DIR / "last.pt")
    if map50 > best_map:
        best_map = map50
        torch.save(ckpt, SAVE_DIR / "best.pt")
        print(f"  * new best model saved (val mAP@50={best_map:.4f})")

total_min = (time.time() - total_start) / 60
print(f"\nTotal training time: {total_min:.1f} min | best val mAP@50: {best_map:.4f}")

## 4. Analysis

### 4.1 Visualize Training Loss vs Epoch


In [ ]:
# Per-epoch summary
for h in history:
    print(f"epoch {h['epoch']:>2}: loss={h['loss']:.4f}  val mAP@50={h['val_map50']:.4f}  ({h['elapsed_sec']:.0f}s)")
print(f"\nBest val mAP@50: {best_map:.4f}")

# Plot training loss and validation mAP@50 vs epoch
epochs_x = [h["epoch"] for h in history]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_x, [h["loss"] for h in history], marker="o")
ax1.set_title("Training loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")

ax2.plot(epochs_x, [h["val_map50"] for h in history], marker="o", color="green")
ax2.set_title("Validation mAP@50")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("mAP@50")

for ax in (ax1, ax2):
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.2 Performance Analysis (Where is the model underperforming)

In [ ]:
# Load the best checkpoint into the model
ckpt = torch.load(SAVE_DIR / "best.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded best.pt (epoch {ckpt['epoch']}, val mAP@50={ckpt['val_map50']:.4f})")

Ground-truth boxes in **green**, predictions (score ≥ 0.5) in **red**. Change `seed` to see other images.

In [ ]:
@torch.no_grad()
def show_gt_vs_pred(model, dataset, device, num_images=6, score_thresh=0.5, seed=0):
    """Draw GT boxes (green) and predictions above score_thresh (red) on random images."""
    rng = random.Random(seed)
    idxs = rng.sample(range(len(dataset)), num_images)
    cols = 3
    rows = math.ceil(num_images / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.5 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, idx in zip(axes, idxs):
        img, target = dataset[idx]
        pred = model([img.to(device)])[0]
        ax.imshow(img.permute(1, 2, 0).cpu().numpy())
        ax.axis("off")

        # Ground truth boxes
        for x1, y1, x2, y2 in target["boxes"].tolist():
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                           fill=False, edgecolor="lime", linewidth=2))
        # Predicted boxes above the threshold
        n_pred = 0
        for box, score in zip(pred["boxes"].cpu(), pred["scores"].cpu()):
            if score < score_thresh:
                continue
            n_pred += 1
            x1, y1, x2, y2 = box.tolist()
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                           fill=False, edgecolor="red", linewidth=2, linestyle="--"))
            ax.text(x1, max(0, y1 - 4), f"{score:.2f}", color="red", fontsize=9,
                    bbox=dict(facecolor="white", alpha=0.6, pad=1))

        ax.set_title(f"val idx {idx} | GT={len(target['boxes'])}  pred={n_pred}")

    for ax in axes[len(idxs):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_gt_vs_pred(model, val_ds, DEVICE, num_images=6, score_thresh=0.5)

### 4.3 Tune the NMS Threshold

NMS removes a box when it overlaps another higher-scoring box by more than this IoU. A higher value helps with overlapping bags, but too high creates duplicates. No retraining needed.

In [ ]:
# Try different NMS IoU thresholds on the best model (no retraining needed)
original_nms = model.roi_heads.nms_thresh
print(f"Default NMS threshold: {original_nms}")

nms_results = {}
for nms in [0.4, 0.5, 0.6, 0.7]:
    model.roi_heads.nms_thresh = nms
    nms_results[nms] = evaluate_map50(model, val_loader, DEVICE)
    print(f"NMS={nms}: val mAP@50={nms_results[nms]:.4f}")

# Keep the best value on the model
best_nms = max(nms_results, key=nms_results.get)
model.roi_heads.nms_thresh = best_nms
print(f"\nBest NMS threshold: {best_nms} (val mAP@50={nms_results[best_nms]:.4f})")

## 5. Test

Speak to a Hackathon Organizer when your team is ready to submit a final model. You will receive two keys to be entered below to download the test dataset and score your model.

### 5.1 Download Test Data

In [ ]:
# Insert keys from Hackathon Organizer
AWS_ACCESS_KEY_ID=""
AWS_SECRET_ACCESS_KEY=""

BUCKET = "uprm-hackathon-data"
PREFIX = "2026/test/"            # trailing slash is optional but recommended
DESTINATION = f"{REPO_ROOT}/data/test/"        # local root folder where files will be saved
REGION = "us-east-1"

In [ ]:
import os
import boto3
from pathlib import Path

# Download
s3 = boto3.resource('s3',
                    aws_access_key_id=AWS_ACCESS_KEY_ID,
                    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
                    region_name=REGION)

bucket = s3.Bucket(BUCKET)

for obj in bucket.objects.filter(Prefix=PREFIX):
    if obj.key.endswith('/'):  # skip “folder” placeholders
        continue
    rel_path = os.path.relpath(obj.key, PREFIX)
    local_path = Path(DESTINATION) / rel_path
    local_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'Downloading {obj.key} → {local_path}')
    bucket.download_file(obj.key, str(local_path))

### 5.2 Create Test DataLoader

In [ ]:
# Create DataLoader for Test Set
test_ds   = KittiPlasticBagDataset(root=DATA_ROOT, split="test")
print(f"test samples: {len(test_ds)}")
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=detection_collate_fn,
    drop_last=False
)

### 5.3 Grade Model on Test Set

In [ ]:
# Load the final model before grading
SAVE_DIR = REPO_ROOT / "models" / "faster_rcnn"
NMS_THRESH = 0.5   # set to the best value found in section 4.3

ckpt = torch.load(SAVE_DIR / "best.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.roi_heads.nms_thresh = NMS_THRESH
model.eval()
print(f"Loaded best.pt (epoch {ckpt['epoch']}, val mAP@50={ckpt['val_map50']:.4f}), NMS={NMS_THRESH}")

In [ ]:
map50 = evaluate_map50(model, test_loader, DEVICE)
print(f"Test mAP@50 (plastic_bag): {map50:.4f}")

# 6. Grading/Submission Guidelines

## Submissions
Please email your submissions to hamzah.abdulrazzaq@lmco.com. All these items must be complete in order for your submission to be considered for evaluation. 
> 1. Email title set to *yourteamname_submission* 
> 2. The python notebook with all your code (.ipynb) 
> 3. Your model checkpoint (.pt) file
> 4. mAP@50 test score copied in with all the digits

**NOTE**: If your email submission is blocked, please try resending the email with a .allow extension added to the file or email title.

## Evaluation Process
> - The submission deadline is on Sunday (9/27) 12PM.
> - Presentations will take place on Sunday (9/27) 2PM for the top 3 teams highest performing solutions. Teams will provide a walkthrough of their code. A winner shall be announced thereafter.